## Reproduction of the paper results

### Decoding: Multivariate Pattern Analysis of EEG Signals

The core analysis in Moerel et al. (2025) uses time-resolved multivariate pattern analysis (MVPA) to determine whether the spatial pattern of EEG activity across 64 channels contains information about the players' decisions at each moment during a trial. Rather than analysing individual channels in isolation, MVPA treats the full set of channel voltages at each time point as a high-dimensional pattern and asks: can a classifier learn to distinguish between Rock, Paper, and Scissors responses based on these patterns? This approach has become standard in cognitive neuroscience because it is sensitive to distributed neural representations that univariate methods would miss (Grootswagers et al., 2017).

#### Phase-wise epoching

Each trial in the experiment consists of three temporally distinct phases: Decision (0–2 s), Response (2–4 s), and Feedback (4–5 s). Critically, the authors do not treat the full 5-second trial as a single epoch for the decoding. Instead, they split each trial into three separate sub-epochs, each locked to the onset of its respective phase:

- **Part A (Decision):** –0.2 to 2.0 s relative to trial onset
- **Part B (Response):** 1.8 to 4.0 s relative to trial onset, then time-shifted so that 0 corresponds to the Response phase onset at 2.0 s
- **Part C (Feedback):** 3.8 to 5.0 s relative to trial onset, then time-shifted so that 0 corresponds to the Feedback phase onset at 4.0 s

The reason for this splitting is that each phase involves a qualitatively different cognitive process: during Decision the participant forms their choice, during Response they execute a button press, and during Feedback they receive visual information about both players' choices. If the entire 5-second epoch were baseline-corrected as a single unit, activity from earlier phases would bleed into the baseline estimate for later phases. By treating each phase independently, the baseline correction can be applied relative to the onset of each phase, ensuring that the decoding results for each phase reflect only the neural processes occurring within that phase.

The 200 ms of overlap (e.g., Part A ends at 2.0 s while Part B starts at 1.8 s) exists specifically to provide each sub-epoch with a pre-phase baseline window of –0.2 to 0 s in its own shifted time frame.

#### Baseline correction

For each of the three sub-epochs, the authors apply baseline correction using the 200 ms window preceding the phase onset (–0.2 to 0 s in the shifted time frame). This involves subtracting the mean voltage across the baseline window from every time point in the sub-epoch, separately for each channel and trial.

Baseline correction serves two purposes here. First, it removes slow voltage drifts that could differ between trials, which would otherwise add noise to the decoding and reduce sensitivity. Second, and more importantly in this context, it ensures that the classifier is decoding *phase-specific* neural activity rather than carry-over activity from the preceding phase. Without per-phase baseline correction, above-chance decoding during the Response phase could partly reflect residual Decision-phase activity rather than genuine Response-phase encoding.

#### Averaging into 250 ms time bins

After baseline correction, the continuous EEG data within each sub-epoch is averaged into non-overlapping 250 ms time bins. This produces 8 bins for the Decision phase (0–2 s), 8 bins for the Response phase (0–2 s in shifted time), and 4 bins for the Feedback phase (0–1 s in shifted time), totalling 20 time bins per trial.

Time-binning serves a dual purpose. First, it reduces the dimensionality of the temporal axis, which makes the subsequent classification more computationally tractable. Second, and more importantly, averaging within 250 ms windows increases the signal-to-noise ratio of each data point by smoothing out high-frequency noise that is unlikely to carry decision-related information. The choice of 250 ms is a standard compromise in the EEG decoding literature: narrow enough to preserve the temporal dynamics of decision-making (which unfolds over hundreds of milliseconds), but wide enough to average out noise effectively. The authors note that they deliberately chose not to apply temporal filtering to the data, as filtering can introduce temporal smearing artifacts that distort the time course of decoded information (van Driel et al., 2021; Delorme, 2023). Time-binning achieves a similar noise-reduction effect without introducing such artifacts.

#### Removal of block-boundary trials

The experiment consisted of 12 blocks of 40 trials each. The first trial of each block was excluded from the decoding analysis for the previous-trial targets (targets 3 and 4), since there is no preceding trial within the same block to provide a valid "previous response" label. This removal is applied to all four decode targets for consistency, resulting in 468 trials per participant (480 minus 12 block-initial trials).

#### Pseudo-trial construction

Before classification, the authors construct pseudo-trials by averaging together small groups of 4 real trials that share the same response class and cross-validation fold. This averaging is repeated 20 times with different random groupings (using CoSMoMVPA's `cosmo_average_samples` function with `'count', 4, 'repeats', 20, 'seed', 1`), producing 20 pseudo-trials per class per fold.

This step is motivated by two considerations. First, averaging increases the signal-to-noise ratio: single EEG trials are extremely noisy, and averaging 4 trials together roughly doubles the SNR (since noise scales with $\sqrt{n}$ while signal scales linearly). This makes the subtle decision-related patterns more detectable by the classifier. Second, pseudo-trial construction naturally balances the number of samples per class. In the raw data, the three response classes (Rock, Paper, Scissors) may have unequal trial counts due to participant biases or no-response trials. After pseudo-trial construction, each class has exactly 20 pseudo-trials per fold, eliminating class imbalance that could bias the classifier.

The balanced sampling scheme (implemented via `cosmo_sample_unique`) ensures that each original trial is used approximately the same number of times across all 20 repeats, so no single trial disproportionately influences the result.

#### Classifier: regularised Linear Discriminant Analysis

The authors use Linear Discriminant Analysis (LDA) with a fixed regularisation parameter of $\lambda = 0.01$ as their classifier (CoSMoMVPA's `cosmo_classify_lda`). LDA models each class as a multivariate Gaussian distribution with a shared covariance matrix and classifies test samples based on which class centroid they are closest to (in Mahalanobis distance). The regularisation adds a small multiple of the identity matrix to the estimated covariance, specifically: $\Sigma_{\text{reg}} = \Sigma + \lambda \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$, where $p$ is the number of features (channels).

LDA is a well-motivated choice for this task for several reasons (Grootswagers et al., 2017; Guggenmos et al., 2018):

1. **Efficiency with small samples:** With ~600 pseudo-trials and 64 features (channels), the sample-to-feature ratio is roughly 10:1. LDA's closed-form solution makes it well-suited for this regime, unlike neural networks which would require considerably more data to avoid overfitting.
2. **Regularisation necessity:** With 64 channels and only ~540 training samples per fold, the raw 64×64 covariance matrix estimate is poorly conditioned. The regularisation stabilises the matrix inversion by shrinking it towards a scaled identity, preventing the classifier from fitting noise in the covariance structure.
3. **Linear assumption:** At the temporal resolution of 250 ms EEG bins, the neural patterns distinguishing Rock, Paper, and Scissors are expected to differ primarily in their mean spatial distributions rather than in complex nonlinear relationships. A linear classifier is therefore appropriate and avoids unnecessary model complexity.
4. **Low computational cost:** LDA is a closed-form solution that requires only computing class means and a shared covariance matrix. This is important because the analysis involves running 20 time bins × 10 folds × 4 decode targets × 62 participants = 49,600 classification runs for the temporal decoding alone, and even more for the searchlight.

#### Cross-validation

Classification is performed using 10-fold cross-validation. The pseudo-trials are divided into 10 folds (chunks) such that each fold contains roughly equal numbers of each response class. In each iteration, the classifier is trained on 9 folds and tested on the remaining fold. Accuracy is computed as the total number of correct predictions across all folds divided by the total number of test samples. With 3 classes, chance-level accuracy is 33.33%.

The chunk assignment is performed by CoSMoMVPA's `cosmo_chunkize` function, which distributes trials across folds in a balanced manner. Importantly, the pseudo-trial averaging is performed *within* each fold (not across folds), so training and test pseudo-trials are constructed from non-overlapping sets of original trials. This prevents information leakage between training and test sets.

#### Decoding targets

The analysis decodes four targets, each providing different information about the decision-making process:

1. **Own response (current trial):** Whether the player chose Rock, Paper, or Scissors. Above-chance decoding indicates that the EEG pattern carries information about the participant's own decision.
2. **Opponent's response (current trial):** Whether the opponent chose Rock, Paper, or Scissors. Above-chance decoding during the Decision and Response phases would suggest that the participant can predict their opponent's move; during Feedback it reflects the visually presented outcome information.
3. **Own previous response:** The player's choice on the preceding trial. Above-chance decoding suggests that the brain maintains a representation of the previous action, which could reflect a strategy (e.g., win-stay, lose-shift).
4. **Opponent's previous response:** The opponent's choice on the preceding trial. Above-chance decoding indicates that the participant encodes the opponent's past behaviour, potentially to inform their current decision.

#### Channel searchlight

In addition to the temporal decoding (which uses all 64 channels), the authors perform a channel searchlight analysis to identify *which brain regions* contribute to the decoding. For each channel, a small neighbourhood is constructed consisting of the channel itself and its 4 nearest neighbours (determined by Euclidean distance between electrode positions), giving 5 features per searchlight location. The same cross-validated LDA decoding is then performed using only this subset of channels. The result is a topographic map of decoding accuracy for each time bin, which can be visualised as a scalp map.

This approach reveals whether the decoded information is driven by localised brain activity (e.g., posterior channels for visual feedback processing) or distributed patterns across the scalp (e.g., for decision-related activity). The paper reports that Decision and Response phase decoding shows distributed topographies, consistent with decision-related processes, while Feedback phase decoding shows a posterior focus, consistent with visual processing of the displayed outcome.

### Our Reproduction of the Decoding Pipeline


The original decoding analysis was implemented in MATLAB using the FieldTrip (version 20240110) and CoSMoMVPA (version 1.1.0) toolboxes. Our reproduction reimplements this pipeline in Python using MNE-Python and NumPy, with custom implementations of several CoSMoMVPA functions where no direct equivalent exists in the Python ecosystem. We describe each step of the reproduction, highlighting where custom code was necessary and what discrepancies we identified along the way.

#### Phase splitting and baseline correction

MNE-Python provides built-in methods for cropping epochs (`epochs.crop()`) and applying baseline correction (`epochs.apply_baseline()`). However, using these directly would not replicate the MATLAB pipeline faithfully. The MATLAB code manually selects time windows from the continuous epoch data using logical indexing on the time vector, shifts the time labels by subtracting the phase onset, and then applies baseline correction on the shifted time axis. MNE's `crop` method, by contrast, operates on the original time axis and does not support the time-shifting step that is needed before baseline correction of Parts B and C.

We therefore implemented the phase splitting and baseline correction manually using NumPy array operations. For each of the three parts (Decision, Response, Feedback), we select the relevant time window from the full epoch array, shift the time labels so that 0 corresponds to the phase onset, and subtract the mean of the –0.2 to 0 s baseline window. This approach matches the MATLAB code line-for-line: the same time boundaries (–0.2 to 2.0 s for Part A, 1.8 to 4.0 s for Part B, 3.8 to 5.0 s for Part C), the same time shifts (subtract 2.0 s for Part B, 4.0 s for Part C), and the same baseline window (–0.2 to 0 s in shifted coordinates).

#### Time binning with strict inequalities

The MATLAB code averages EEG data within each 250 ms bin using strict inequalities: `time > t_start & time < t_end`. This means that time points falling exactly on a bin boundary are excluded. We replicated this behaviour exactly in Python (`(times > window[0]) & (times < window[1])`). Using non-strict inequalities (`>=` and `<=`) would include boundary samples in two adjacent bins, subtly altering the averaged values. This seems like a minor detail, but at 256 Hz sampling rate the bin edges occasionally coincide with sample times, so the distinction matters for exact replication.

#### Custom LDA classifier

This was the most critical implementation decision in the entire reproduction. Scikit-learn provides `LinearDiscriminantAnalysis` with a `shrinkage` parameter, which would appear to be a straightforward replacement for CoSMoMVPA's `cosmo_classify_lda`. However, after consulting the CoSMoMVPA source code, we discovered that the two implementations use fundamentally different regularisation formulas.

CoSMoMVPA computes the regularised covariance as:

$$\Sigma_{\text{reg}} = \Sigma + \lambda \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$$

where $\Sigma$ is the pooled within-class covariance normalised by $N$ (the total number of training samples), $\lambda = 0.01$ is the regularisation parameter, $p$ is the number of features, and $I$ is the identity matrix. This is an **additive** regularisation: the identity matrix scaled by the average eigenvalue is added to the covariance.

Scikit-learn's shrinkage LDA, by contrast, uses a **convex combination**:

$$\Sigma_{\text{reg}} = (1 - \lambda) \cdot \Sigma + \lambda \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$$

With $\lambda = 0.01$, the difference is small (the first formula gives $\Sigma + 0.01 \cdot \mu \cdot I$ while the second gives $0.99 \cdot \Sigma + 0.01 \cdot \mu \cdot I$, where $\mu = \frac{\text{trace}(\Sigma)}{p}$), but it is nonzero and accumulates across the 49,600+ classification runs in the full analysis. Additionally, CoSMoMVPA normalises the pooled covariance by the total number of training samples $N$, whereas scikit-learn normalises by $N – K$ (where $K$ is the number of classes).

We therefore wrote a custom LDA implementation that matches the CoSMoMVPA source code exactly:

1. Compute per-class means
2. Compute pooled within-class scatter matrix: $S_w = \sum_k (X_k - \mu_k)^T (X_k - \mu_k)$
3. Normalise by N (total training samples): $\Sigma = \frac{S_w}{N}$
4. Add regularisation: $\Sigma_{\text{reg}} = \Sigma + 0.01 \cdot \frac{\text{trace}(\Sigma)}{p} \cdot I$
5. Compute class weights: $W = \mu \cdot \Sigma_{\text{reg}}^{-1}$ (via `np.linalg.solve` for numerical stability)
6. Compute bias: $b_k = \sum_j W_{kj} \cdot \mu_{kj}$
7. Predict: $\hat{y} = \arg\max_k \left( x \cdot W_k^T - 0.5 \cdot b_k \right)$

This is worth discussing because it illustrates a broader point about reproduction studies: even when two implementations claim to perform "regularised LDA," the specific regularisation formula can differ between toolboxes, and these differences can affect results. In our case, using scikit-learn's default shrinkage would have produced similar but not identical results to the original paper.

#### Custom `cosmo_chunkize` implementation

CoSMoMVPA's `cosmo_chunkize` function assigns cross-validation fold labels in a balanced manner. Given that each trial starts with a unique chunk label (as set by the MATLAB code: `ds_sel.sa.chunks = (1:numel(ds_sel.sa.targets))'`), the function groups these individual trials into 10 balanced folds through an optimisation procedure that minimises class imbalance across folds.

For the specific case where every input chunk contains exactly one sample (which is always the case in this pipeline), the optimisation reduces to a simple round-robin assignment: for each target class, trials are assigned sequentially to folds 1, 2, ..., 10, 1, 2, ..., and so on. We implemented this deterministic assignment directly, which matches the CoSMoMVPA output for single-trial input chunks.

Importantly, this is *not* the same as scikit-learn's `StratifiedKFold`, which shuffles trial order before splitting (when `shuffle=True`) or uses a different sequential partitioning strategy (when `shuffle=False`). Using `StratifiedKFold` would produce different fold compositions and therefore different decoding accuracies.

#### Custom `cosmo_sample_unique` for balanced pseudo-trial averaging

CoSMoMVPA's `cosmo_average_samples` function internally calls `cosmo_sample_unique` to generate balanced sampling indices. This function ensures that across all 20 repeats, each original trial is used approximately the same number of times — unlike simple random sampling, where some trials might be used many times while others are never selected.

The algorithm works by:

1. Generating (repeats + 1) random permutations of the trial indices and concatenating them into a flat vector
2. Walking through this vector to fill each repeat column, skipping indices that have already been used in the current column or that have already been visited globally
3. Sorting each column of the resulting index matrix

We ported this algorithm directly from the CoSMoMVPA MATLAB source code to Python, including the column-major flattening (`ravel(order='F')`) to match MATLAB's memory layout. A simpler approach — calling `rng.choice(indices, size=count, replace=False)` independently for each repeat — would not guarantee balanced usage across repeats and would produce different pseudo-trials.

#### Random number generator differences

One unavoidable difference between our Python reproduction and the MATLAB original is the random number generator. MATLAB's `rng(seed)` uses the Mersenne Twister algorithm with a specific seeding convention, while NumPy's `default_rng(seed)` uses the PCG64 generator. Even with the same seed value, the two generators produce entirely different sequences of random numbers.

This affects `cosmo_sample_unique` (which random groupings of trials are averaged together for pseudo-trial construction) and therefore the exact pseudo-trial values. The statistical properties are identical — both produce balanced, approximately uniform sampling — but the specific pseudo-trials differ between the MATLAB and Python pipelines. As a consequence, the per-participant decoding accuracies will differ slightly between the two implementations, though the group-level patterns should converge. This is an inherent limitation of cross-language reproduction that cannot be resolved without reimplementing MATLAB's Mersenne Twister in Python, which we considered unnecessary given that the statistical equivalence is preserved.

#### Cross-validation and accuracy computation

The 10-fold cross-validation loop is straightforward to reproduce: iterate over the 10 chunk labels, hold out one chunk as the test set, train on the remaining 9, and accumulate correct predictions. We compute accuracy as total correct divided by total test samples across all folds, matching CoSMoMVPA's `cosmo_crossvalidation_measure` with `'output', 'accuracy'`.

#### Channel searchlight

For the channel searchlight, the original MATLAB code uses `cosmo_meeg_chan_neighborhood(ds, 'count', 4)` to define a neighbourhood of exactly 4 nearest channels for each searchlight centre. We replicate this by computing a pairwise Euclidean distance matrix from the electrode positions (obtained from the MNE montage), then for each channel selecting the 4 channels with the smallest distances. Combined with the centre channel itself, this gives 5 features per searchlight location, matching the original.

One subtlety here is the coordinate space. The MNE montage stores electrode positions in real-world metres (scaled from the biosemi64.mat unit-circle coordinates by a factor of 0.09, corresponding to a 9 cm head radius). Since we select neighbours by rank order (the 4 closest) rather than by a distance threshold, the absolute scale of the coordinates does not affect the neighbour selection — the ranking is preserved under any uniform scaling. This differs from the preprocessing step, where the neighbour computation for channel interpolation uses a distance threshold and therefore *is* sensitive to the coordinate scale (see the preprocessing section for details on this issue).

#### Behavioural data construction

The behavioural matrices (mapping each trial to the player's response, the opponent's response, the outcome, and the previous-trial responses for both players) were constructed following the MATLAB code's logic exactly. One implementation detail worth noting is the player swap: in the raw BDF files, the channel prefixes "1-" and "2-" are reversed relative to the behavioural data labels "Player 1" and "Player 2." The MATLAB preprocessing script handles this by assigning prefix "2-" channels to Player 1 and prefix "1-" channels to Player 2. Our Python preprocessing replicates this swap, and the decoding script constructs the same 5-column behavioural matrix for each player with identical outcome recoding for Player 2 (swapping win/loss codes so that the outcome is always relative to the current player).

#### Summary of reproduction fidelity

The following table summarises the correspondence between the MATLAB and Python implementations:

| Step | MATLAB (FieldTrip / CoSMoMVPA) | Python (our implementation) | Match |
|------|------|------|------|
| Phase splitting | `ft_selectdata` with `cfg.latency` | NumPy boolean masking on time vector | Exact |
| Time shifting | Manual assignment of shifted time vectors | Subtraction on time arrays | Exact |
| Baseline correction | `ft_preprocessing` with `cfg.demean` | Manual mean subtraction on NumPy arrays | Exact |
| Time binning | Manual loop with strict `>` and `<` | Manual loop with strict `>` and `<` | Exact |
| Trial removal | `rem_idx = 1:40:480` | `np.arange(0, 480, 40)` (0-indexed) | Exact |
| Chunk assignment | `cosmo_chunkize(ds, 10)` | Custom `cosmo_chunkize` (round-robin) | Exact for single-trial chunks |
| Pseudo-trial averaging | `cosmo_average_samples` | Custom `cosmo_sample_unique` + averaging | Algorithmic match; RNG differs |
| LDA classifier | `cosmo_classify_lda` (λ=0.01) | Custom implementation (additive reg, Σ/N) | Exact formula match |
| Cross-validation | `cosmo_nfold_partitioner` + `cosmo_crossvalidation_measure` | Chunk-based leave-one-out loop | Exact |
| Searchlight neighbours | `cosmo_meeg_chan_neighborhood('count', 4)` | 4 nearest by Euclidean distance | Exact (rank-based) |
| Re-referencing | `ft_preprocessing` with `cfg.reref='yes'` | `epochs.set_eeg_reference('average')` | Exact |

The only source of non-determinism between the two pipelines is the random number generator used for pseudo-trial construction. All algorithmic choices, formulas, and parameters are matched to the original MATLAB code.

## Our Contributions
*decoding --> (felipe) linear decoding*